In [88]:
import random

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
%matplotlib inline

words = open("../src/data/names.txt", 'r').read().splitlines()

In [89]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}; stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [126]:
def build_dataset(words):
    block_size = 4
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size
        for c in w + '.':
            ix = stoi[c]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [127]:
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[:n2])

In [69]:
# --------------------------------------------------------------------------------------
# E01: Tune the hyperparameters of the training to beat AK's best validation loss of 2.2
# --------------------------------------------------------------------------------------

In [70]:
# Well I already did in the main learning notebook, but I'll make it even better... with blackjack and hookers

In [114]:
g = torch.Generator().manual_seed(2147483647-10)
C = torch.randn(27, 30, generator=g)
W1 = torch.randn(120, 300, generator=g)
b1 = torch.randn(300, generator=g)
W2 = torch.randn(300, 27, generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

45237

In [79]:
max_step = 200_000
lrs = torch.linspace(0.1, 0.01, max_step)

for i in range(max_step):
    # mini-batch
    ix = torch.randint(0, Xtr.shape[0], (100,))
    # forward
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 120) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    # backward
    for p in parameters:
        p.grad = None
    loss.backward()
    # nudge
    lr = 0.1 if i < 100_000 else 0.01
    # lr = lrs[i]
    for p in parameters:
        p.data += -lr * p.grad

print(loss.item())

1.8778518438339233


In [80]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 120) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.1477, grad_fn=<NllLossBackward0>)

In [50]:
# --------------------------------------------------------------------------------------
# E02: I was not careful with the intialization of the network in this video.
# (1) What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform?
#     What loss do we achieve?
# (2) Can you tune the initialization to get a starting loss that is much more similar to (1)?
# --------------------------------------------------------------------------------------

In [128]:
g = torch.Generator().manual_seed(2147483647-10)
C = torch.randn(27, 30)
W1 = torch.ones(120, 300)
b1 = torch.randn(300, generator=g)
W2 = torch.ones(300, 27)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [129]:
max_step = 200_000
lrs = torch.linspace(0.1, 0.01, max_step)

for i in range(max_step):
    # mini-batch
    ix = torch.randint(0, Xtr.shape[0], (100,))
    # forward
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 120) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    # backward
    for p in parameters:
        p.grad = None
    loss.backward()
    # nudge
    lr = 0.1 if i < 100_000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

print(loss.item())

1.7979075908660889


In [130]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 120) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.0193, grad_fn=<NllLossBackward0>)

In [ ]:
# --------------------------------------------------------------------------------------
# E03: Read the Bengio et al 2003 paper, implement and try any idea from the paper. Did it work?
# --------------------------------------------------------------------------------------

In [16]:
# https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf